# What sits between the users and the model servers, and what does each part buy?

Question: when users call model servers through a gateway, what do batch, real-time and streaming modes, routing, failover, disaggregation, cold-start fixes, queues, autoscaling and the right Triton batcher each buy? The notebook answers in numbers.

Why the exam asks: objectives inference.4, inference.1 and inference.3 name these parts. Three questions were missed: what Dynamo adds to an engine, which Triton batcher fits a stateful model, and what shortens a slow NIM start. The notebook shows the mechanism behind each one.

Setup:

- Kaggle GPU T4 x2, Internet on. Each T4 has 15 GiB. It has no bf16 and no NVLink. The two GPUs share one host. All model runs use fp16.
- Model: `Qwen/Qwen2.5-1.5B-Instruct` with vLLM, one replica on each T4. The cell 1 output shows the vLLM version.
- The kernel never imports vllm or torch. vLLM runs as separate processes. The gateway, the load generator and the simulations are plain Python in this kernel.
- Every part runs through `part(name, fn, budget)`. A failed part prints its traceback and the run goes on. The last cell prints a status table.
- Every request, server start and install has a timeout. Expected run time: 40 to 50 minutes. The notebook starts no new part after 52 minutes (parts F and G need no GPU and always run).
- `NVIDIA_API_KEY` is optional. With it, part C adds a hosted model as a third provider (about 4 calls). Without it, that step is skipped. The key is never printed or saved.
- Seeds are fixed. Rates carry Wilson 95% intervals. Two systems on the same items use a paired bootstrap or an exact McNemar test.
- The log holds every table. Each table is also saved as CSV in `/kaggle/working`, with raw rows as JSON and plots as PNG.
- The client runs on the same machine as the servers. It uses threads and one warm connection per thread. Part A checks that the client is not the bottleneck.

Objectives: inference.4 (batch, real-time and streaming services), inference.1 (gateways and routing, provider abstraction, Dynamo, autoscaling, request queuing, cold starts, scaling strategies), inference.3 (disaggregated serving, and the other optimization terms, listed at the end).

Terms used below. Sources are files in `nvidia-nca-amad-exam/hub/sources`. Terms without a source file are general ideas that the notebook measures.

- **Model server**: a process that loads a model and answers requests. Here it is vLLM.
- **Gateway**: one endpoint in front of many model servers. It picks a server for each request. It can retry and check health.
- **Routing policy**: the rule that picks the server. *Round robin* (rr) takes the servers in turn. *Least outstanding requests* (lor) takes the server with the fewest unfinished requests. *Prefix-aware* routing, also called KV-aware, takes the server that most likely holds the KV cache of the start of the prompt.
- **Provider abstraction**: one client call and one request format over many back ends. The gateway hides the address, the model name and the key of each back end.
- **NVIDIA Dynamo**: the engine runs the model on a GPU. Dynamo runs the fleet of engines. It adds disaggregated serving, KV cache-aware routing, KV cache offload across memory tiers and SLA-based autoscaling. Dynamo runs on top of engines such as vLLM, SGLang and TensorRT-LLM. Sources: `introduction.txt` (Dynamo docs v1.3.0), `agentic-inference.txt`.
- **KV cache and prefix caching**: the engine keeps the keys and values of earlier tokens. A new prompt with the same start reuses them and skips that part of prefill. Source: `introduction.txt`.
- **Prefill and decode**: prefill reads the prompt and builds the KV cache. Decode makes one token per step and is limited by memory bandwidth. Source: `mastering-llm-techniques-inference-optimization.txt`.
- **TTFT and ITL**: time to first token, and the gap between two tokens.
- **Disaggregated serving**: prefill and decode run on different devices, so each can be tuned and scaled alone. It needs a fast way to move the KV cache. NIXL is the transfer layer of Dynamo. vLLM has a `NixlConnector` that uses it. Source: `introduction.txt`.
- **Cold start**: the time from the start of a pod to its first answer. A readiness probe tells the orchestrator that a pod can take traffic.
- **Model cache**: NVIDIA recommends to cache the model once, on a persistent volume with network storage, so that many pods share it. Source: `cache.txt` (NIM Operator).
- **Request queue and load shedding**: a request waits when all slots are busy. When the queue is full, shedding rejects the new request at once.
- **Autoscaling**: the number of replicas follows the load. The NIM example scales a Kubernetes deployment on the KV cache utilization of the pods. Source: `horizontal-autoscaling-of-nvidia-nim-microservices-on-kubernetes.txt`. A **warm pool** is a set of spare replicas that are already loaded.
- **Triton dynamic batcher and sequence batcher**: the dynamic batcher is for stateless models and spreads batches over all instances. The sequence batcher is for stateful models and sends all requests of one sequence to one instance. Source: `batcher.txt`.
- **Batch, real-time and streaming**: batch sends all requests at once and asks only for throughput. Real-time answers each request with a full reply. Streaming sends the reply token by token.

## Predict before running

Write your own guess first. Then compare it with the tables at the end.

| Part | What you measure | Your guess |
|---|---|---|
| A | Tokens/s, TTFT and end-to-end p50 and p95 for batch, real-time and streaming. Which mode wins on which number? | |
| B | TTFT and cached prompt tokens for round robin, least outstanding and prefix-aware routing. By how much does prefix-aware routing cut TTFT on turns 2 to 4? | |
| C | Failed requests when replica 1 dies: no retry, retry, retry with health checks. | |
| D | Inter-token latency of chats when long prompts arrive: mixed, smaller chunks, roles on two GPUs, KV connector. Which setup helps most? Does the KV connector run on two T4s? | |
| E | Seconds to the first answer: no local weights, cached weights, warm page cache. Which phase shrinks? How many requests does an early ready probe lose? | |
| F | Latency and shed share in a traffic spike: fixed replicas, shedding, autoscaling with cold start, cached weights and a warm pool. | |
| G | Wrong-state answers per 1,000 windows: dynamic batcher against sequence batcher on a stateful model. | |

## Setup and tools

Cells 1 to 5 build the tools and test them. No GPU part runs before the tests pass.

- Cell 1 installs vLLM in a subprocess and prints the GPUs.
- Cell 2 holds the part runner, statistics, the `/metrics` parser, the HTTP client and the server control. The client keeps one warm connection per thread. It starts the clock after it connects.
- Cell 3 is the gateway: routing policies, retry, health checks and provider records.
- Cell 4 holds three pure-Python tools: the queue and autoscaling simulator, the Triton batcher simulator and the parser of the vLLM start log.
- Cell 5 runs eight self-checks. Each one tests a tool against a known answer: a fixed Prometheus text, a known Wilson interval, a stream whose first text arrives 0.3 s after the role chunk, the routing rules, a gateway with one broken provider, three queue formulas, the Triton truth, and a fixed vLLM log text.

In [ ]:
# 1. Install vLLM and check the GPUs. This kernel never imports vllm or torch, so nothing here is upgraded under a loaded package.
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vllm"], capture_output=True, text=True, timeout=900)
print("pip install vllm: exit code", r.returncode, r.stderr[-300:] if r.returncode else "")
print(subprocess.run([sys.executable, "-c", "import vllm; print('vllm', vllm.__version__)"], capture_output=True, text=True, timeout=300).stdout)
print(subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total,compute_cap", "--format=csv"], capture_output=True, text=True, timeout=60).stdout)

In [ ]:
# 2. Helpers: part runner, secret, statistics, metrics parser, HTTP client, server control. No GPU use, no vllm import.
import atexit, glob, heapq, http.client, json, math, os, random, re, shutil, signal, subprocess, sys, threading, time, traceback, urllib.request, uuid
from collections import deque
from concurrent.futures import ThreadPoolExecutor
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.stats import binomtest

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
OUT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
PORT, GPU = dict(r1=8001, r2=8002), dict(r1=0, r2=1)
T0, GLOBAL_LIMIT_S, STATUS, DEADLINE = time.time(), 52 * 60, [], [0.0]
try:
    from kaggle_secrets import UserSecretsClient
    KEY = UserSecretsClient().get_secret("NVIDIA_API_KEY")
except Exception:
    KEY = os.environ.get("NVIDIA_API_KEY", "")
print("NVIDIA_API_KEY present:", bool(KEY))              # the value is never printed or saved

def scrub(s): return str(s).replace(KEY, "***") if KEY else str(s)
def late(): return time.time() > DEADLINE[0]
def part(name, fn, budget_s, always=False):
    """Run one part. A failure prints its traceback, is recorded, and the notebook goes on."""
    t = time.time(); DEADLINE[0] = t + budget_s; state = "ok"
    if not always and t - T0 > GLOBAL_LIMIT_S: state = "skipped: global time limit"
    else:
        try: fn()
        except Exception: state = "FAILED"; print(scrub(traceback.format_exc())[-3000:])
        if state == "ok" and late(): state = "over budget"
    STATUS.append(dict(part=name, state=state, seconds=round(time.time() - t), budget_s=budget_s))
    print(f"[{(time.time() - T0) / 60:5.1f} min] part {name}: {state}")

def show(df, name):
    df.to_csv(f"{OUT}/{name}.csv", index=False); print(df.to_string(index=False, line_width=190))
def raw(rows, name):
    json.dump(rows, open(f"{OUT}/{name}.json", "w"), default=str)

# ---- statistics ----
def pct(xs, p):
    xs = [x for x in xs if x == x]
    return float(np.percentile(xs, p)) if xs else float("nan")
def wilson(k, n, z=1.96):
    if n == 0: return float("nan"), float("nan")
    p, d = k / n, 1 + z * z / n
    c, h = (p + z * z / (2 * n)) / d, z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return max(0.0, c - h), min(1.0, c + h)
def rate(k, n):
    lo, hi = wilson(k, n); return f"{k}/{n} = {k / max(n, 1):.1%} [{lo:.1%}, {hi:.1%}]"
def paired_boot(a, b, n=2000, seed=0):
    """Mean of a - b and a 95% paired bootstrap interval. a and b have equal length and are paired by position."""
    d, rng = np.asarray(a, float) - np.asarray(b, float), np.random.default_rng(seed)
    m = [rng.choice(d, len(d)).mean() for _ in range(n)]
    return d.mean(), np.percentile(m, 2.5), np.percentile(m, 97.5)
def mcnemar(a_ok, b_ok):
    """Exact McNemar test on two lists of True/False for the same items. Returns the two discordant counts and the p value."""
    x, y = sum(p and not q for p, q in zip(a_ok, b_ok)), sum(q and not p for p, q in zip(a_ok, b_ok))
    return x, y, (binomtest(min(x, y), x + y, 0.5).pvalue if x + y else 1.0)

# ---- Prometheus text from vLLM /metrics ----
def parse_metrics(text):
    m = {}
    for l in text.splitlines():
        if not l or l[0] == "#": continue
        k, _, v = l.rpartition(" "); name = k.split("{", 1)[0]
        if name.endswith(("_bucket", "_created")): continue
        try: m[name] = m.get(name, 0.0) + float(v)          # sum over label sets (engines)
        except ValueError: pass
    return m
def metrics(port): return parse_metrics(urllib.request.urlopen(f"http://127.0.0.1:{port}/metrics", timeout=10).read().decode())
def mget(m, *names): return next((m[n] for n in names if n in m), 0.0)
HITS, QUERIES = ("vllm:prefix_cache_hits_total", "vllm:gpu_prefix_cache_hits_total"), ("vllm:prefix_cache_queries_total", "vllm:gpu_prefix_cache_queries_total")

# ---- HTTP client: one warm connection per thread and target ----
_tl = threading.local()
def getconn(host, port, tls=False):
    d = _tl.__dict__.setdefault("c", {}); k = (host, port, tls)
    if k not in d: d[k] = (http.client.HTTPSConnection if tls else http.client.HTTPConnection)(host, port, timeout=300)
    return d[k]
def dropconn(host, port, tls=False):
    c = _tl.__dict__.setdefault("c", {}).pop((host, port, tls), None)
    if c: c.close()

def chat(port, messages, max_tokens=64, stream=True, ignore_eos=True, host="127.0.0.1", extra=None):
    """One chat request. TTFT is the time to the first chunk with text in it: the first chunk only has the role."""
    body = dict(model=MODEL, messages=messages, max_tokens=max_tokens, temperature=0, ignore_eos=ignore_eos, **(extra or {}))
    if stream: body.update(stream=True, stream_options={"include_usage": True})
    res = dict(outcome="ok", err="", ttft=None, itl=[], tokens=0, text="", e2e=None)
    for attempt in (0, 1):
        try:
            c = getconn(host, port)
            if c.sock is None: c.connect()                  # connect before the clock starts, so every request uses a warm connection
            t0 = time.perf_counter()
            c.request("POST", "/v1/chat/completions", json.dumps(body), {"Content-Type": "application/json"})
            r = c.getresponse()
            if r.status != 200: raise RuntimeError(f"HTTP {r.status} {r.read()[:100]!r}")
            if stream:
                last, chunks = None, 0
                for line in r:
                    line = line.strip()
                    if not line.startswith(b"data:") or line.endswith(b"[DONE]"): continue
                    ev = json.loads(line[5:]); ch = ev.get("choices")
                    if ev.get("usage"): res["tokens"] = ev["usage"]["completion_tokens"]
                    piece = ch[0]["delta"].get("content") if ch else None
                    if piece:
                        now = time.perf_counter(); res["text"] += piece; chunks += 1
                        if res["ttft"] is None: res["ttft"] = now - t0
                        else: res["itl"].append(now - last)
                        last = now
                res["tokens"] = res["tokens"] or chunks
            else:
                d = json.load(r); res["tokens"] = d["usage"]["completion_tokens"]; res["text"] = d["choices"][0]["message"]["content"]
            res["e2e"] = time.perf_counter() - t0
            break
        except Exception as e:
            dropconn(host, port)
            stale = isinstance(e, (http.client.RemoteDisconnected, BrokenPipeError, ConnectionResetError)) and not res["text"]
            if attempt == 0 and stale: continue            # an idle warm connection that the server closed: retry once
            res.update(outcome="error", err=scrub(repr(e))[:150]); break
    if res["outcome"] == "ok" and ignore_eos and res["tokens"] < max_tokens: res["outcome"] = "short"    # cut: fewer tokens than asked
    return res

def load(reqs, workers, deadline_s=600, on_done=None):
    """Send requests (chat() arguments plus optional at=start offset in s and tag=label) from a thread pool. Returns (results, wall s)."""
    t0, out, n = time.perf_counter(), [None] * len(reqs), [0]
    def job(i):
        r = dict(reqs[i]); at, tag = r.pop("at", 0), r.pop("tag", "")
        time.sleep(max(0, at - (time.perf_counter() - t0)))
        res = chat(**r) if time.perf_counter() - t0 < deadline_s else dict(outcome="not_reached", err="", ttft=None, itl=[], tokens=0, text="", e2e=None)
        res["tag"] = tag; out[i] = res; n[0] += 1
        if on_done: on_done(n[0])
    with ThreadPoolExecutor(workers) as ex: list(ex.map(job, range(len(reqs))))
    return out, time.perf_counter() - t0

def summ(rs, wall):
    ok, c = [r for r in rs if r["outcome"] == "ok"], pd.Series([r["outcome"] for r in rs]).value_counts()
    return dict(n=len(rs), ok=len(ok), short=int(c.get("short", 0)), error=int(c.get("error", 0)), not_reached=int(c.get("not_reached", 0)),
                tok_s=sum(r["tokens"] for r in rs if r["outcome"] in ("ok", "short")) / wall,
                ttft_p50=pct([r["ttft"] * 1000 for r in ok if r["ttft"] is not None], 50), ttft_p95=pct([r["ttft"] * 1000 for r in ok if r["ttft"] is not None], 95),
                e2e_p50=pct([r["e2e"] * 1000 for r in ok], 50), e2e_p95=pct([r["e2e"] * 1000 for r in ok], 95))

# ---- text of a chosen token length: a seeded word list, sized with the server's /tokenize ----
WORDS = ("time year people way day man thing woman life child world school state family student group country problem hand part place case week company "
         "system program question work government number night point home water room mother area money story fact month lot right study book eye job word "
         "business issue side kind head house service friend father power hour game line end member law car city community name team minute idea body "
         "information back face level office door health person art war history party result change morning reason research moment air force education").split()
_TPW = []
def make_text(n_tokens, seed):
    if not _TPW:
        s = " ".join(random.Random(0).choices(WORDS, k=400)); _TPW.append(1.3)
        for port in (8001, 8002, 8101, 8102, 8100, 8200):
            try:
                rq = urllib.request.Request(f"http://127.0.0.1:{port}/tokenize", json.dumps({"model": MODEL, "prompt": s}).encode(), {"Content-Type": "application/json"})
                _TPW[0] = json.load(urllib.request.urlopen(rq, timeout=20))["count"] / 400; print(f"tokens per filler word: {_TPW[0]:.2f} (from port {port})"); break
            except Exception: pass
        else: print("no /tokenize answer: using 1.3 tokens per word")
    return " ".join(random.Random(seed).choices(WORDS, k=max(1, round(n_tokens / _TPW[0]))))

# ---- vLLM server control. A server runs in its own process group, so one kill stops the engine workers too. ----
PROCS, GPU_OF = {}, {}
def gpu_mib():
    try: return [int(x) for x in subprocess.check_output(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"], text=True, timeout=20).split()]
    except Exception: return [0, 0]
def wait_gpu_free(gpu, limit=90):
    t = time.time()
    while time.time() - t < limit and gpu_mib()[gpu] > 1500: time.sleep(2)
def launch(name, gpu, port, *args, env=None):
    wait_gpu_free(gpu)
    cmd = ["vllm", "serve", MODEL, "--dtype", "half", "--max-model-len", "4096", "--gpu-memory-utilization", "0.85", "--max-num-seqs", "64", "--port", str(port), *args]
    GPU_OF[name] = gpu
    PROCS[name] = subprocess.Popen(cmd, stdout=open(f"{OUT}/{name}.log", "w"), stderr=subprocess.STDOUT, start_new_session=True,
                                   env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), **(env or {})))
def log_tail(name, n=2500): return scrub(open(f"{OUT}/{name}.log").read()[-n:])
def wait_ready(name, port, limit=600):
    t = time.time()
    while time.time() - t < limit:
        if PROCS[name].poll() is not None: raise RuntimeError(f"{name} exited. Log tail:\n{log_tail(name)}")
        try: urllib.request.urlopen(f"http://127.0.0.1:{port}/health", timeout=3).close(); return time.time() - t
        except Exception: time.sleep(1)
    raise RuntimeError(f"{name} not ready after {limit} s. Log tail:\n{log_tail(name)}")
def kill(name, sig=signal.SIGKILL):
    p = PROCS.get(name)
    if p and p.poll() is None:
        try: os.killpg(p.pid, sig)
        except ProcessLookupError: pass
def stop(*names):
    names = [n for n in names if n in PROCS]
    for n in names: kill(n, signal.SIGTERM)
    for n in names:
        try: PROCS[n].wait(timeout=60)
        except Exception: kill(n); PROCS[n].wait()
    for g in {GPU_OF[n] for n in names}: wait_gpu_free(g)
def ensure(*names, wait=True):
    """Start the replicas that are not running. r1 uses GPU 0 and r2 uses GPU 1."""
    for n in names:
        if n not in PROCS or PROCS[n].poll() is not None: launch(n, GPU[n], PORT[n])
    for n in names:
        if wait: print(f"{n} ready after {wait_ready(n, PORT[n]):.0f} s")
atexit.register(lambda: [kill(n) for n in PROCS])

In [ ]:
# 3. The gateway: one OpenAI-style endpoint in front of several providers. Three routing policies, retry, health checks.
# A provider is a dict: name, host, port, and optional tls, model, key, extra (body keys to add), keep (body keys to keep), fallback (use only when no other provider is left).
HOSTED = dict(name="hosted", host="integrate.api.nvidia.com", port=443, tls=True, model="nvidia/nemotron-3.5-lightning-30b-a3b", key=KEY,
              extra={"chat_template_kwargs": {"enable_thinking": False}}, keep=("model", "messages", "max_tokens", "temperature", "stream"), fallback=True)
LOAD_SLACK, BLOCK = 4, 256     # prefix policy: how many more outstanding requests the cache owner may have; characters per hashed block

class Fail(Exception):
    def __init__(self, msg, started=False): super().__init__(msg); self.started = started   # started: bytes already went to the client

def post_chat(p, body, headers=None):
    """Send one chat request to provider p on this thread's warm connection. Returns the response. Raises Fail."""
    b = dict(body, model=p.get("model", body["model"]), **p.get("extra", {}))
    if "keep" in p: b = {k: v for k, v in b.items() if k in p["keep"]}
    hdr = {"Content-Type": "application/json", **({"Authorization": "Bearer " + p["key"]} if p.get("key") else {}), **(headers or {})}
    for attempt in (0, 1):
        try:
            c = getconn(p["host"], p["port"], bool(p.get("tls"))); c.request("POST", "/v1/chat/completions", json.dumps(b), hdr)
            r = c.getresponse()
            if r.status >= 500: raise OSError(f"HTTP {r.status}")
            return r
        except (OSError, http.client.HTTPException) as e:
            dropconn(p["host"], p["port"], bool(p.get("tls")))
            if attempt == 0 and isinstance(e, (http.client.RemoteDisconnected, BrokenPipeError, ConnectionResetError)): continue   # the server closed an idle warm connection
            raise Fail(scrub(repr(e)))

def relay(h, p, body, headers=None):
    """Copy the reply of provider p to the client handler h. A stream is copied as it arrives."""
    r = post_chat(p, body, headers)
    ct = r.getheader("Content-Type", "application/json")
    if not body.get("stream") or r.status != 200:
        try: data = r.read()
        except (OSError, http.client.HTTPException) as e: dropconn(p["host"], p["port"], bool(p.get("tls"))); raise Fail(repr(e))
        h.send_response(r.status); h.send_header("Content-Type", ct); h.send_header("Content-Length", str(len(data))); h.end_headers(); h.wfile.write(data); return
    h.send_response(200); h.send_header("Content-Type", ct); h.send_header("Transfer-Encoding", "chunked"); h.end_headers()
    while True:
        try: d = r.read1(8192)
        except (OSError, http.client.HTTPException) as e: dropconn(p["host"], p["port"], bool(p.get("tls"))); raise Fail(repr(e), True)
        if not d: break
        h.wfile.write(b"%x\r\n%s\r\n" % (len(d), d)); h.wfile.flush()
    h.wfile.write(b"0\r\n\r\n")

def block_hashes(messages):
    """Chained hash of each full block of the prompt text. Two prompts with the same start share their first hashes."""
    s, h, out = "".join(f"<{m['role']}>{m['content']}" for m in messages), 0, []
    for i in range(0, len(s) - BLOCK + 1, BLOCK): h = hash((h, s[i:i + BLOCK])); out.append(h)
    return out

class Srv(ThreadingHTTPServer): request_queue_size = 128      # the default backlog of 5 would delay new connections under load

class GW(BaseHTTPRequestHandler):
    protocol_version = "HTTP/1.1"; disable_nagle_algorithm = True
    def log_message(self, *a): pass
    def reply_error(self, code, msg):
        d = json.dumps({"error": msg}).encode(); self.send_response(code); self.send_header("Content-Length", str(len(d))); self.end_headers(); self.wfile.write(d)
    def do_POST(self):
        body, gw, tried = json.loads(self.rfile.read(int(self.headers["Content-Length"]))), self.server.gw, set()
        while True:
            p = gw.pick(body, tried)
            if p is None: return self.reply_error(503, "no provider left")
            try: relay(self, p, body); gw.done(p, True); return
            except Fail as e:
                gw.done(p, False)
                if e.started: self.close_connection = True; return          # the stream is already half sent: it cannot be retried
                if not gw.retry: return self.reply_error(502, str(e))
                tried.add(p["name"]); gw.retries += 1

class Gateway:
    """policy: rr (round robin), lor (least outstanding requests), prefix (KV-aware: send a prompt to the provider that most likely holds its prefix)."""
    def __init__(self, providers, policy="rr", retry=False, health=False):
        self.P, self.policy, self.retry, self.health = providers, policy, retry, health
        self.lock, self.rr, self.retries, self.stopping = threading.Lock(), 0, 0, threading.Event()
        names = [p["name"] for p in providers]
        self.out, self.count, self.up, self.seen = dict.fromkeys(names, 0), dict.fromkeys(names, 0), dict.fromkeys(names, True), {n: set() for n in names}
        self.srv = Srv(("127.0.0.1", 0), GW); self.srv.gw = self
        self.port = self.srv.server_address[1]
        threading.Thread(target=self.srv.serve_forever, daemon=True).start()
        if health: threading.Thread(target=self.poll, daemon=True).start()
    def poll(self):
        while not self.stopping.wait(0.3):
            for p in self.P:
                if p.get("tls"): continue
                try: urllib.request.urlopen(f"http://{p['host']}:{p['port']}/health", timeout=1).close(); self.up[p["name"]] = True
                except Exception: self.up[p["name"]] = False
    def pick(self, body, tried):
        with self.lock:
            c = [p for p in self.P if p["name"] not in tried and (self.up[p["name"]] or not self.health)]
            c = [p for p in c if not p.get("fallback")] or c                 # a fallback provider only when no other one is left
            if not c: return None
            load = lambda p: self.out[p["name"]]
            if self.policy == "rr": p = c[self.rr % len(c)]; self.rr += 1
            elif self.policy == "lor": p = min(c, key=load)
            else:
                hs = block_hashes(body["messages"])
                def match(p): return next((i for i, h in enumerate(hs) if h not in self.seen[p["name"]]), len(hs))
                best = max(c, key=lambda p: (match(p), -load(p)))
                p = best if match(best) and load(best) <= min(map(load, c)) + LOAD_SLACK else min(c, key=load)   # no cache hit, or the owner is too busy: least loaded
                self.seen[p["name"]].update(hs)       # ponytail: the router's own record of what it sent, not the engine's real cache. Dynamo listens to cache events instead.
            self.out[p["name"]] += 1
            return p
    def done(self, p, ok):
        with self.lock:
            self.out[p["name"]] -= 1
            if ok: self.count[p["name"]] += 1
            elif self.health: self.up[p["name"]] = False        # passive check: the health poller marks it up again
    def close(self): self.stopping.set(); self.srv.shutdown(); self.srv.server_close()

In [ ]:
# 4. Pure-Python simulators and the startup-log parser: queue and autoscaling (part F), Triton batchers (part G), cold-start phases (part E).
def simulate(arr, svc, slots, n0, qmax=None, scale=None):
    """Queue simulation. A replica has `slots` parallel slots. A request waits first in, first out. A full queue (qmax) sheds the request.
    scale = dict(tick, thresh, delay, max, pool, warm): every `tick` s, if more than `thresh` requests wait, start one more replica.
    It is ready after `delay` s. A spare replica from the warm pool (`pool`) is ready after `warm` s. Returns latency per request (wait + service), shed flags, cost."""
    N, H = len(arr), arr[-1]
    ev = [(t, 0, i) for i, t in enumerate(arr)]; heapq.heapify(ev)
    q, lat, shed = deque(), np.full(N, np.nan), np.zeros(N, bool)
    free, nrep, pend, pool = n0 * slots, n0, 0, (scale or {}).get("pool", 0)
    rep_s, last, peak = 0.0, 0.0, 0
    if scale: heapq.heappush(ev, (scale["tick"], 3, -1))
    def start(j, t): lat[j] = t - arr[j] + svc[j]; heapq.heappush(ev, (t + svc[j], 1, j))
    while ev:
        t, k, i = heapq.heappop(ev)
        rep_s += (nrep + pend + pool) * (min(t, H) - min(last, H)); last = t        # a starting replica and a spare one cost money too
        if k == 0:                                           # arrival
            if free: free -= 1; start(i, t)
            elif qmax is None or len(q) < qmax: q.append(i)
            else: shed[i] = True
            peak = max(peak, len(q))
        elif k == 1:                                         # departure: the slot takes the next waiting request
            if q: start(q.popleft(), t)
            else: free += 1
        elif k == 2:                                         # a new replica is ready
            nrep += 1; pend -= 1
            for _ in range(slots):
                if q: start(q.popleft(), t)
                else: free += 1
        elif t < H or q or pend:                             # autoscaler tick
            if len(q) > scale["thresh"] and nrep + pend < scale["max"]:
                pend += 1
                if pool: pool -= 1; heapq.heappush(ev, (t + scale["warm"], 2, -1))
                else: heapq.heappush(ev, (t + scale["delay"], 2, -1))
            heapq.heappush(ev, (t + scale["tick"], 3, -1))
    return dict(lat=lat, shed=shed, peak=peak, replica_s=rep_s, replicas=nrep)

def erlang_wait(lam, mu, c):
    """Mean wait in an M/M/c queue (Erlang C)."""
    a = lam / mu; top = a ** c / math.factorial(c) * c / (c - a)
    return top / (sum(a ** k / math.factorial(k) for k in range(c)) + top) / (c * mu - lam)

ALPHA = 0.3
def triton_truth(v):
    """Running score of one machine: an exponentially weighted mean of its windows, from the first window on."""
    out, s = [], None
    for x in v: s = x if s is None else ALPHA * x + (1 - ALPHA) * s; out.append(s)
    return out

def triton_sim(batcher, stateful=True, machines=40, windows=25, inst=2, max_batch=8, per_step=24, seed=0):
    """Two model instances behind a Triton batcher. Windows of 40 machines arrive interleaved, in order per machine.
    A stateful model keeps each machine's running score inside the instance. The dynamic batcher gives a batch to any free instance.
    The sequence batcher sends every window of one sequence (one correlation ID) to one instance and puts one window per sequence in a batch.
    A wrong answer is an output that differs from the single-instance truth. A stateless model outputs 2 * value."""
    rng = np.random.default_rng(seed)
    val = rng.normal(size=(machines, windows))
    truth = np.array([triton_truth(v) for v in val]) if stateful else 2 * val
    nxt, reqs = np.zeros(machines, int), []
    for m in rng.permutation(np.repeat(np.arange(machines), windows)): reqs.append((m, nxt[m])); nxt[m] += 1
    pending, state, owner, active = deque(reqs), [dict() for _ in range(inst)], {}, [0] * inst
    seq = batcher == "sequence"; queues = [deque() for _ in range(inst if seq else 1)]
    wrong = steps = batches = rows = 0
    while pending or any(queues):
        for _ in range(min(per_step, len(pending))):
            m, w = pending.popleft()
            if seq:
                if m not in owner: owner[m] = int(np.argmin(active)); active[owner[m]] += 1          # sequence START: pick an instance
                queues[owner[m]].append((m, w))
            else: queues[0].append((m, w))
        for i in range(inst):
            q, batch = queues[i if seq else 0], []
            if seq:
                skipped = deque()
                while q and len(batch) < max_batch:
                    r = q.popleft(); (skipped if r[0] in {b[0] for b in batch} else batch).append(r)
                q.extendleft(reversed(skipped))
            else: batch = [q.popleft() for _ in range(min(max_batch, len(q)))]
            if not batch: continue
            batches += 1; rows += len(batch); snap = dict(state[i])         # every row of a batch reads the state from before the batch
            for m, w in batch:
                x, prev = val[m, w], (snap.get(m) if w else None)
                out = (x if prev is None else ALPHA * x + (1 - ALPHA) * prev) if stateful else 2 * x
                state[i][m] = out; wrong += abs(out - truth[m, w]) > 1e-9
                if seq and w == windows - 1: active[owner[m]] -= 1; state[i].pop(m)         # sequence END: free the slot
        steps += 1
    return dict(wrong=int(wrong), windows=len(reqs), batch=rows / batches, steps=steps)

# ---- vLLM startup log: seconds per phase. A phase that the log does not print stays NaN. ----
PHASES = {"download_s": r"Time spent downloading weights.*?: ([\d.]+) seconds", "weights_load_s": r"Model loading took .*?([\d.]+) seconds",
          "compile_s": r"torch\.compile takes ([\d.]+) s", "graph_capture_s": r"Graph capturing finished in ([\d.]+) sec",
          "engine_init_s": r"init engine .*? took ([\d.]+) seconds"}
def parse_startup(text):
    out = {k: (sum(map(float, re.findall(p, text))) if re.search(p, text) else float("nan")) for k, p in PHASES.items()}
    kv, mem = re.search(r"GPU KV cache size: ([\d,]+) tokens", text), re.search(r"Available KV cache memory: ([\d.]+) GiB", text)
    out["kv_tokens"], out["kv_gib"] = (int(kv.group(1).replace(",", "")) if kv else float("nan")), (float(mem.group(1)) if mem else float("nan"))
    return out

In [ ]:
# 5. Self-checks. Each one tests a piece of code against a known answer, so a wrong result later cannot hide a broken tool. They run first.
def check_metrics_parser():
    m = parse_metrics('# HELP x\nvllm:prefix_cache_hits_total{engine="0",model_name="a b"} 120.0\nvllm:prefix_cache_hits_total{engine="1",model_name="a b"} 30\n'
                      'vllm:num_requests_waiting 2\nx_bucket{le="1"} 9\n')
    assert m == {"vllm:prefix_cache_hits_total": 150.0, "vllm:num_requests_waiting": 2.0}, m

def check_wilson():
    lo, hi = wilson(0, 10); assert lo == 0 and abs(hi - 0.2775) < 1e-3, (lo, hi)      # known interval for 0 of 10
    d, lo, hi = paired_boot([3, 4, 5, 6], [1, 2, 3, 4]); assert d == 2 and lo == hi == 2

def check_ttft():
    """A stream whose first chunk has only the role. The first text arrives 0.3 s later. TTFT must be about 0.3 s, not 0."""
    class H(BaseHTTPRequestHandler):
        protocol_version = "HTTP/1.1"
        def log_message(self, *a): pass
        def do_POST(self):
            self.rfile.read(int(self.headers["Content-Length"])); self.send_response(200); self.send_header("Content-Type", "text/event-stream")
            self.send_header("Connection", "close"); self.end_headers()
            def ev(d): self.wfile.write(b"data: " + (json.dumps(d).encode() if d != "[DONE]" else d.encode()) + b"\n\n"); self.wfile.flush()
            ev({"choices": [{"delta": {"role": "assistant", "content": ""}}]}); time.sleep(0.3)
            ev({"choices": [{"delta": {"content": "Hi"}}]}); time.sleep(0.1); ev({"choices": [{"delta": {"content": " there"}}]})
            ev({"choices": [], "usage": {"completion_tokens": 2}}); ev("[DONE]")
    s = ThreadingHTTPServer(("127.0.0.1", 0), H); threading.Thread(target=s.serve_forever, daemon=True).start()
    r = chat(s.server_address[1], [{"role": "user", "content": "x"}], max_tokens=2)
    s.shutdown(); s.server_close()
    assert r["outcome"] == "ok" and r["tokens"] == 2 and r["text"] == "Hi there", r
    assert 0.28 <= r["ttft"] < 0.45, r["ttft"]
    assert 0.08 <= r["itl"][0] < 0.25, r["itl"]

def check_router():
    P, msg = [dict(name="a", host="x", port=1), dict(name="b", host="x", port=2)], {"messages": []}
    g = Gateway(P, "rr"); order = [g.pick(msg, set())["name"] for _ in range(4)]; g.close(); assert order == ["a", "b", "a", "b"], order
    g = Gateway(P, "lor"); g.out["a"] = 3; assert g.pick(msg, set())["name"] == "b"; g.close()
    g = Gateway(P, "prefix"); m1 = {"messages": [{"role": "system", "content": "d" * 3000}, {"role": "user", "content": "q1"}]}
    first = g.pick(m1, set())["name"]; other = "b" if first == "a" else "a"
    m2 = {"messages": m1["messages"] + [{"role": "assistant", "content": "a1"}, {"role": "user", "content": "q2"}]}
    g.out[first] = 2; assert g.pick(m2, set())["name"] == first, "a small load gap must keep the cache owner"
    g.out[first] = 9; assert g.pick(m2, set())["name"] == other, "an overloaded cache owner must lose"
    assert g.pick({"messages": [{"role": "user", "content": "e" * 3000}]}, set()) is not None; g.close()

def check_failover():
    """Provider 'bad' closes every connection with no reply. Provider 'good' answers. Without retry, half of the requests fail. With retry, none."""
    def serve(good):
        class H(BaseHTTPRequestHandler):
            protocol_version = "HTTP/1.1"
            def log_message(self, *a): pass
            def do_POST(self):
                self.rfile.read(int(self.headers["Content-Length"]))
                if not good: self.close_connection = True; return
                d = json.dumps({"choices": [{"message": {"content": "ok"}}], "usage": {"completion_tokens": 1}}).encode()
                self.send_response(200); self.send_header("Content-Type", "application/json"); self.send_header("Content-Length", str(len(d))); self.end_headers(); self.wfile.write(d)
        s = ThreadingHTTPServer(("127.0.0.1", 0), H); threading.Thread(target=s.serve_forever, daemon=True).start(); return s
    bad, good = serve(False), serve(True); errors = {}
    P = [dict(name="bad", host="127.0.0.1", port=bad.server_address[1]), dict(name="good", host="127.0.0.1", port=good.server_address[1])]
    for retry in (False, True):
        g = Gateway(P, "rr", retry=retry)
        errors[retry] = sum(chat(g.port, [{"role": "user", "content": "x"}], 1, stream=False, ignore_eos=False)["outcome"] == "error" for _ in range(20)); g.close()
    for s in (bad, good): s.shutdown(); s.server_close()
    assert errors == {False: 10, True: 0}, errors

def check_sim():
    rng, N = np.random.default_rng(0), 300_000
    lam, mu = 0.7, 1.0                                                                     # M/M/1: mean wait = lam / (mu * (mu - lam))
    arr, svc = np.cumsum(rng.exponential(1 / lam, N)), rng.exponential(1 / mu, N)
    w, f = np.mean(simulate(arr, svc, 1, 1)["lat"] - svc), lam / (mu * (mu - lam))
    assert abs(w - f) / f < 0.07, ("M/M/1 mean wait", w, f)
    lam, c = 6.4, 8                                                                        # M/M/8, Erlang C
    arr, svc = np.cumsum(rng.exponential(1 / lam, N)), rng.exponential(1 / mu, N)
    w, f = np.mean(simulate(arr, svc, c, 1)["lat"] - svc), erlang_wait(lam, mu, c)
    assert abs(w - f) / f < 0.07, ("M/M/8 mean wait", w, f)
    lam, qmax = 0.9, 4                                                                     # M/M/1/K with K = 1 + qmax: blocking probability
    arr, svc = np.cumsum(rng.exponential(1 / lam, N)), rng.exponential(1 / mu, N)
    shed, rho, K = simulate(arr, svc, 1, 1, qmax=qmax)["shed"].mean(), lam / mu, qmax + 1
    f = (1 - rho) * rho ** K / (1 - rho ** (K + 1))
    assert abs(shed - f) < 0.01, ("blocking probability", shed, f)

def check_triton():
    assert np.allclose(triton_truth([0.0, 10.0, 10.0]), [0.0, 3.0, 5.1])
    dyn, seq, stl = triton_sim("dynamic"), triton_sim("sequence"), triton_sim("dynamic", stateful=False)
    assert dyn["wrong"] > 100 and seq["wrong"] == 0 and stl["wrong"] == 0 and seq["batch"] > 1.5, (dyn, seq, stl)

def check_startup_parser():
    t = ("INFO Time spent downloading weights for Q/m: 12.500000 seconds\nINFO Model loading took 2.89 GiB and 3.2 seconds\nINFO torch.compile takes 7.4 s in total\n"
         "INFO Graph capturing finished in 8 secs, took 0.45 GiB\nINFO init engine (profile, create kv cache, warmup model) took 25.9 seconds\n"
         "INFO GPU KV cache size: 262,144 tokens\nINFO Available KV cache memory: 7.5 GiB\n")
    assert parse_startup(t) == dict(download_s=12.5, weights_load_s=3.2, compile_s=7.4, graph_capture_s=8.0, engine_init_s=25.9, kv_tokens=262144, kv_gib=7.5)
    assert math.isnan(parse_startup("nothing here")["download_s"])

def selfchecks():
    fails = []
    for f in (check_metrics_parser, check_wilson, check_ttft, check_router, check_failover, check_sim, check_triton, check_startup_parser):
        t = time.time()
        try: f(); print(f"PASS {f.__name__} ({time.time() - t:.1f} s)")
        except AssertionError as e: fails.append(f.__name__); print(f"FAIL {f.__name__}: {scrub(e)[:300]}")
    assert not fails, f"self-checks failed: {fails}"
part("0 self-checks", selfchecks, 120, always=True)

## Part A: batch, real time and streaming (inference.4)

Question: for the same 200 requests, what does each way of serving give you? Why the exam asks: the best practice differs by mode. Batch wants throughput. Real-time wants a low end-to-end time. Streaming wants a low time to the first token.

The 200 prompts are short notes. Each reply has exactly 128 tokens (`ignore_eos`), so the work is equal in all modes.

- **Offline batch**: `LLM.chat` on all 200 prompts at once, in a separate process on GPU 1, with the same engine settings (64 sequences at once). It has no per-request time. Only the wall time counts.
- **Real-time, full reply**: one HTTP call per request, no streaming. Concurrency 8 (a typical interactive load) and 64 (the same number of running sequences as the batch).
- **Streaming**: the same calls with `stream=true`. TTFT is the time to the first chunk **with text**. The first chunk of a chat stream only holds the role, so it does not count. Inter-token latency is the gap between text chunks.

The client check compares the mean TTFT of the client with the mean TTFT that vLLM reports itself, counts the CPU time of the client, and measures how many chunks per second one core can parse.

In [ ]:
# 6. Part A: the same 200 requests as an offline batch, as real-time calls with a full reply, and as token streams.
N_REQ, MAXTOK = 200, 128
OFFLINE = '''import json, sys, time
from vllm import LLM, SamplingParams
msgs, model, mt = json.load(open(sys.argv[1])), sys.argv[2], int(sys.argv[3])
llm = LLM(model=model, dtype="half", max_model_len=4096, gpu_memory_utilization=0.85, max_num_seqs=64)
sp = SamplingParams(temperature=0, max_tokens=mt, ignore_eos=True)
llm.chat(msgs[:8], sp)
t = time.perf_counter(); outs = llm.chat(msgs, sp); wall = time.perf_counter() - t
print("RESULT", json.dumps({"wall": wall, "tokens": sum(len(o.outputs[0].token_ids) for o in outs)}))
'''
A_E2E = []      # end-to-end seconds of the real-time run at c=8: part F uses them as measured service times

def parse_capacity(n=20000):
    """How many stream chunks per second one core of this client can parse. This is the client ceiling."""
    line = b'data: ' + json.dumps({"choices": [{"delta": {"content": " word"}}]}).encode() + b'\n'
    t = time.perf_counter()
    for _ in range(n):
        ev = json.loads(line.strip()[5:]); ev["choices"][0]["delta"].get("content")
    return n / (time.perf_counter() - t)

def part_a():
    global A_E2E
    pairs = zip(random.Random(1).choices(WORDS, k=N_REQ), random.Random(2).choices(WORDS, k=N_REQ))
    msgs = [[{"role": "user", "content": f"Write a short note about {a} and {b}. Use plain words."}] for a, b in pairs]
    rows = []
    ensure("r1", wait=False)                                  # replica 1 starts on GPU 0 while the offline batch runs on GPU 1
    json.dump(msgs, open(f"{OUT}/a_msgs.json", "w")); open(f"{OUT}/offline.py", "w").write(OFFLINE)
    r = subprocess.run([sys.executable, f"{OUT}/offline.py", f"{OUT}/a_msgs.json", MODEL, str(MAXTOK)], capture_output=True, text=True, timeout=480,
                       env=dict(os.environ, CUDA_VISIBLE_DEVICES="1"))
    line = next((l for l in r.stdout.splitlines() if l.startswith("RESULT ")), None)
    if line:
        o = json.loads(line[7:]); rows.append(dict(mode="offline batch (LLM.chat, 200 at once)", n=N_REQ, ok=N_REQ, tok_s=o["tokens"] / o["wall"], wall_s=o["wall"]))
    else: print("offline batch did not run. stderr tail:\n" + scrub(r.stderr[-1500:]))
    ensure("r1", "r2")
    port = PORT["r1"]; chat(port, msgs[0], 8)                  # warm-up: kernels and connection
    reqs = lambda **kw: [dict(port=port, messages=m, max_tokens=MAXTOK, **kw) for m in msgs]
    for label, kw, c in (("real-time, full reply, c=8", dict(stream=False), 8), ("real-time, full reply, c=64", dict(stream=False), 64), ("streaming, c=8", dict(stream=True), 8)):
        if late(): print("budget used, stop before", label); break
        if kw["stream"]: m0, cpu0 = metrics(port), time.process_time()
        res, wall = load(reqs(**kw), c); s = summ(res, wall); rows.append(dict(mode=label, wall_s=wall, **s)); raw([{k: v for k, v in x.items() if k != "text"} for x in res], f"a_raw_{len(rows)}")
        if label.endswith("full reply, c=8"): A_E2E = [x["e2e"] for x in res if x["outcome"] == "ok"]
        if kw["stream"]:
            m1, cpu = metrics(port), (time.process_time() - cpu0) / wall
            srv = (mget(m1, "vllm:time_to_first_token_seconds_sum") - mget(m0, "vllm:time_to_first_token_seconds_sum")) / max(1, mget(m1, "vllm:time_to_first_token_seconds_count") - mget(m0, "vllm:time_to_first_token_seconds_count")) * 1000
            cli = np.mean([x["ttft"] * 1000 for x in res if x["ttft"] is not None]); head = parse_capacity() / s["tok_s"]
            print(f"client check: mean TTFT client {cli:.1f} ms, server {srv:.1f} ms, gap {cli - srv:.1f} ms | client CPU {cpu:.2f} cores | parse ceiling {head:.0f}x the chunk rate")
            ok = (cli - srv) < max(10, 0.1 * cli) and cpu < 0.7 and head > 5
            print("the client is " + ("not the bottleneck (gap under 10 ms or 10%, under 0.7 core, ceiling above 5x)" if ok else "a possible bottleneck: read the three numbers above"))
    df = pd.DataFrame(rows); show(df.round(1), "a_modes")
    row = lambda name: df[df["mode"] == name].iloc[0]
    if {"streaming, c=8", "real-time, full reply, c=8"} <= set(df["mode"]):
        print(f"\nhighest tokens/s: {df.loc[df.tok_s.idxmax(), 'mode']} ({df.tok_s.max():.0f} tok/s).")
        st, rt = row("streaming, c=8"), row("real-time, full reply, c=8")
        print(f"streaming: first text after {st.ttft_p50:.0f} ms (p50). The same request as a full reply returns after {rt.e2e_p50:.0f} ms (p50) at c=8.")
part("A batch, real-time, streaming", part_a, 480)

## Part B: routing in front of two replicas (inference.1, Dynamo)

Question: does it matter which replica gets a request? Why the exam asks: this is the missed question about Dynamo. An engine runs the model on one GPU. Dynamo adds the layer above it, and KV cache-aware routing is one part of that layer (`introduction.txt`).

The gateway in cell 3 is about 100 lines of Python. It has three policies:

- **rr**: replicas in turn.
- **lor**: the replica with the fewest unfinished requests.
- **prefix**: the gateway hashes the prompt in blocks of 256 characters and remembers which replica it sent each block to. A prompt goes to the replica with the longest match, unless that replica has 4 or more unfinished requests than the least loaded one. This is a simple form of KV-aware routing. Dynamo's router uses real cache events from the engines. This one only uses its own record.

The load: 12 sessions with 4 turns each. Each turn adds a question and the real reply to the chat, so a later turn starts with the full earlier text. Four documents of about 2,400 tokens, each shared by three sessions. Sessions start 0.4 s apart. Every policy runs twice with new documents each time, so no policy meets a cache that an earlier run filled. The cached tokens come from the `/metrics` page of both replicas.

In [ ]:
# 7. Part B: three routing policies in front of two replicas. 12 sessions, 4 turns each. Four groups of three sessions share one long document.
SESSIONS, TURNS, DOC_TOK, POLICIES = 12, 4, 2400, ("rr", "lor", "prefix")

def run_sessions(gw_port, seed):
    res = []
    def session(s):
        time.sleep(0.4 * s)                                    # sessions start one after the other, so the first prefill of a document can finish
        msgs = [{"role": "system", "content": "Answer questions about this document in one sentence.\n\n" + make_text(DOC_TOK, seed * 100 + s % 4)}]
        for t in range(TURNS):
            msgs.append({"role": "user", "content": f"Question {t + 1} from user {s}: what is the main idea of the first paragraph?"})
            r = chat(gw_port, msgs, max_tokens=40, ignore_eos=False); r.update(session=s, turn=t); res.append(r)
            msgs.append({"role": "assistant", "content": r["text"] or "ok"})
    with ThreadPoolExecutor(SESSIONS) as ex: list(ex.map(session, range(SESSIONS)))
    return res

def part_b():
    ensure("r1", "r2"); ports, runs = [PORT["r1"], PORT["r2"]], {}
    P = [dict(name=n, host="127.0.0.1", port=PORT[n]) for n in ("r1", "r2")]
    for rep in (1, 2):
        for i, pol in enumerate(POLICIES):
            if late(): print("budget used, stop before", pol, rep); break
            gw = Gateway(P, pol); m0 = [metrics(p) for p in ports]
            res = run_sessions(gw.port, seed=rep * 10 + i); m1 = [metrics(p) for p in ports]; gw.close()
            hit = sum(mget(b, *HITS) - mget(a, *HITS) for a, b in zip(m0, m1)); qry = sum(mget(b, *QUERIES) - mget(a, *QUERIES) for a, b in zip(m0, m1))
            runs[(rep, pol)] = dict(res=res, hit=hit, qry=qry, count=dict(gw.count))
            raw([{k: v for k, v in x.items() if k not in ("text", "itl")} | dict(rep=rep, policy=pol) for x in res], f"b_raw_rep{rep}_{pol}")
    def ttft(pol, only2=False):      # {(rep, session, turn): TTFT ms}
        return {(rep, x["session"], x["turn"]): x["ttft"] * 1000 for (rep, p), r in runs.items() if p == pol for x in r["res"] if x["ttft"] is not None and (x["turn"] > 0 or not only2)}
    rows = []
    for pol in POLICIES:
        mine = [r for (_, p), r in runs.items() if p == pol]
        if not mine: continue
        rs, hit, qry = [x for r in mine for x in r["res"]], sum(r["hit"] for r in mine), sum(r["qry"] for r in mine)
        served = sum(sum(r["count"].values()) for r in mine)
        rows.append(dict(policy=pol, requests=len(rs), errors=sum(x["outcome"] == "error" for x in rs), ttft_p50=pct(list(ttft(pol).values()), 50),
                         ttft_p95=pct(list(ttft(pol).values()), 95), ttft_turn2plus_p50=pct(list(ttft(pol, True).values()), 50), cached_tokens=hit,
                         cache_hit_rate=hit / qry if qry else float("nan"), hit_ci95="[%.3f, %.3f]" % wilson(hit, qry), share_on_r1=sum(r["count"].get("r1", 0) for r in mine) / max(1, served),
                         r1_ci95="[%.2f, %.2f]" % wilson(sum(r["count"].get("r1", 0) for r in mine), served)))
    df = pd.DataFrame(rows); show(df.round(3), "b_routing")
    base = ttft("rr", True)
    for pol in ("lor", "prefix"):
        t, k = ttft(pol, True), [x for x in ttft(pol, True) if x in base]
        if k:
            d, lo, hi = paired_boot([t[x] for x in k], [base[x] for x in k])
            print(f"{pol} minus rr, TTFT of turns 2 to 4, paired by (rep, session, turn): mean {d:+.0f} ms, 95% interval [{lo:+.0f}, {hi:+.0f}] ms, {len(k)} pairs")
    if len(df) == 3:
        w = df.loc[df.ttft_turn2plus_p50.idxmin()]
        print(f"lowest TTFT p50 on turns 2 to 4: {w.policy} ({w.ttft_turn2plus_p50:.0f} ms). Highest cache hit rate: {df.loc[df.cache_hit_rate.idxmax(), 'policy']} ({df.cache_hit_rate.max():.0%}).")
    if len(df):
        df.set_index("policy")[["ttft_p50", "ttft_p95", "ttft_turn2plus_p50"]].plot.bar(ylabel="TTFT ms", title="TTFT by routing policy"); plt.tight_layout(); plt.savefig(f"{OUT}/b_routing.png"); plt.close()
part("B routing", part_b, 300)

## Part C: one client over many providers, and failover (inference.1)

Question: what happens to the users when a replica dies, and what do retry and health checks change? Why the exam asks: a gateway is also the place where you hide the provider and survive failures.

1. **Provider abstraction.** The same `chat()` call goes through one gateway per provider: replica 1, replica 2 and, with a key, a hosted model (`nvidia/nemotron-3.5-lightning-30b-a3b`, thinking off). The provider record holds the address, the model name, the key and the body rules. The client code does not change.
2. **Failover.** 120 non-streaming requests, 8 at a time, round robin over two replicas. After 30 requests are done, replica 1 gets `SIGKILL`. Three gateway settings run the same requests. *Retry*: when a provider fails before any byte reaches the client, the gateway tries the next one. *Health check*: the gateway polls `/health` every 0.3 s and drops a failed provider from the choice. Replica 1 restarts between the runs.
3. **Fallback to the hosted model** (only with a key): with replica 1 dead, the hosted provider answers because it is the last one left.

In [ ]:
# 8. Part C: one client call over replica 1, replica 2 and a hosted model. Then kill replica 1 in the middle of a run.
def part_c():
    ensure("r1", "r2")
    local = [dict(name=n, host="127.0.0.1", port=PORT[n]) for n in ("r1", "r2")]
    hello = [{"role": "user", "content": "Say hello in five words."}]
    # provider abstraction: the same chat() call. Only the gateway behind it differs, and the gateway hides the model name, the key and the body rules.
    rows = []
    for p in local + ([HOSTED] if KEY else []):
        g = Gateway([p]); r = chat(g.port, hello, 24, stream=False, ignore_eos=False); g.close()
        rows.append(dict(provider=p["name"], model_sent=p.get("model", MODEL), outcome=r["outcome"], e2e_ms=(r["e2e"] or 0) * 1000, reply=(r["text"] or r["err"])[:50].replace("\n", " ")))
    if not KEY: print("no NVIDIA_API_KEY: the hosted provider is skipped")
    show(pd.DataFrame(rows).round(0), "c_providers")
    # failover: kill replica 1 after 30 of 120 requests are done. Three gateway settings, the same requests.
    msgs = [[{"role": "user", "content": f"Name one fact about {w}. Answer in one sentence."}] for w in random.Random(5).choices(WORDS, k=120)]
    rows, oks = [], {}
    for label, retry, health in (("no retry, no health check", False, False), ("retry only", True, False), ("retry + health check", True, True)):
        if late(): print("budget used, stop before", label); break
        ensure("r1", "r2")                                      # restart replica 1 when the last run killed it
        gw = Gateway(local, "rr", retry, health); fired = []
        def on_done(n):
            if n == 30 and not fired: fired.append(time.time()); kill("r1")
        res, wall = load([dict(port=gw.port, messages=m, max_tokens=48, stream=False, ignore_eos=False) for m in msgs], 8, on_done=on_done)
        raw([{k: v for k, v in x.items() if k not in ("text", "itl")} | dict(setting=label) for x in res], f"c_raw_{len(rows)}")
        oks[label] = [x["outcome"] == "ok" for x in res]; bad = sum(not o for o in oks[label]); e = [x["e2e"] * 1000 for x in res if x["outcome"] == "ok"]
        rows.append(dict(setting=label, failed=rate(bad, len(res)), retries=gw.retries, served_r1=gw.count["r1"], served_r2=gw.count["r2"], e2e_p50_ms=pct(e, 50), e2e_p95_ms=pct(e, 95), mcnemar_vs_first=""))
        if label != "no retry, no health check" and "no retry, no health check" in oks:
            x, y, pv = mcnemar(oks["no retry, no health check"], oks[label]); rows[-1]["mcnemar_vs_first"] = f"{x} vs {y}, p={pv:.3g}"
        gw.close()
    show(pd.DataFrame(rows).round(0), "c_failover")
    print("served_r1 counts requests that replica 1 finished before the kill. mcnemar: requests that failed only in the first setting against only in this one.")
    if KEY and not late():                                      # both local replicas are gone or busy: the hosted provider is the last one left
        g = Gateway([local[0], HOSTED], "rr", True, True)       # replica 1 is dead after the last run
        r = [chat(g.port, hello, 24, stream=False, ignore_eos=False) for _ in range(3)]; g.close()
        print(f"replica 1 dead, hosted provider as fallback: {sum(x['outcome'] == 'ok' for x in r)}/3 answered, served by {dict(g.count)}")
    stop("r1", "r2")
part("C providers and failover", part_c, 480)

## Part D: prefill and decode interference, and disaggregated serving (inference.3)

Question: what happens to chats that are in the middle of a reply when a long prompt arrives? Why the exam asks: prefill is compute-bound and decode is memory-bound. On one GPU they compete. Disaggregated serving puts them on different devices (`introduction.txt`).

The load: 8 chat streams of 160 tokens each, one after another, for 26 s. Sixteen long prompts of 3,000 tokens arrive 1.5 s apart. The numbers are the inter-token latency (ITL) of the chats, and the TTFT of the long prompts.

| Setup | What it is |
|---|---|
| chats only | The floor: no long prompts. |
| mixed, budget 8192 | One server. A step can hold up to 8,192 tokens, so a long prefill fits in one step. |
| mixed, budget 512 | One server. A prefill is cut into pieces of at most 512 tokens (chunked prefill). |
| roles split | Chats on GPU 0, long prompts on GPU 1. No KV cache moves. This removes the interference, but it is not disaggregated serving: the prefill GPU never hands a KV cache to a decode GPU. |
| KV connector | Real disaggregation. GPU 0 prefills, GPU 1 decodes. vLLM moves the KV cache with `NixlConnector` (NIXL). A small proxy sends each request to the producer first, and then to the consumer with the transfer parameters. |

The KV connector try has a limit of 12 minutes. The two T4s have no NVLink and share one host. If the connector does not run, the notebook prints the real error. That is a valid result. The vLLM page on this feature says the connector is experimental. The KV connector rows go through the proxy, which adds one hop that the other rows do not have.

In [ ]:
# 9. Part D: long prompts arrive while chats decode. Mixed on one server, smaller chunks, roles on two GPUs, and a real KV connector.
WIN, N_LONG, CHATS = 26, 16, 8       # seconds of chat load, long prompts (3000 tokens, one every 1.5 s), steady chat streams
D_ROWS = []

def interf(label, chat_port, long_port, seed):
    """CHATS steady chat streams (160 tokens each, one after the other) and N_LONG long prompts. ITL is the gap between two text chunks of a chat."""
    t0, chats, longs = time.perf_counter(), [], []
    def chat_loop(k):
        i = 0
        while time.perf_counter() - t0 < WIN:
            chats.append(chat(chat_port, [{"role": "user", "content": make_text(60, seed * 1000 + k * 60 + i)}], max_tokens=160)); i += 1
    def long_job(i):
        time.sleep(max(0, 2 + 1.5 * i - (time.perf_counter() - t0)))
        longs.append(chat(long_port, [{"role": "user", "content": make_text(3000, seed * 1000 + 900 + i)}], max_tokens=8))
    with ThreadPoolExecutor(CHATS + N_LONG) as ex:
        fs = [ex.submit(chat_loop, k) for k in range(CHATS)] + ([ex.submit(long_job, i) for i in range(N_LONG)] if long_port else [])
        for f in fs: f.result()
    gaps = [g * 1000 for c in chats for g in c["itl"]]; lt = [x["ttft"] * 1000 for x in longs if x["ttft"] is not None]
    D_ROWS.append(dict(setup=label, chat_gaps=len(gaps), itl_p50=pct(gaps, 50), itl_mean=float(np.mean(gaps)) if gaps else float("nan"), itl_p95=pct(gaps, 95), itl_p99=pct(gaps, 99), itl_max=max(gaps, default=float("nan")),
                       stalls_over_100ms=sum(g > 100 for g in gaps) / max(1, len(gaps)), long_ttft_p50=pct(lt, 50), long_ttft_p95=pct(lt, 95),
                       errors=sum(x["outcome"] == "error" for x in chats + longs)))
    raw([{k: v for k, v in x.items() if k != "text"} for x in chats + longs], f"d_raw_{len(D_ROWS)}")
    r = D_ROWS[-1]; print(f"{label}: ITL p50 {r['itl_p50']:.1f} ms, p99 {r['itl_p99']:.1f} ms, max {r['itl_max']:.0f} ms, {r['errors']} errors")

class PD(BaseHTTPRequestHandler):
    """Toy prefill/decode proxy: the producer prefills, the consumer gets the KV cache and decodes. The same steps as vLLM's toy_proxy_server."""
    protocol_version = "HTTP/1.1"; disable_nagle_algorithm = True
    def log_message(self, *a): pass
    def do_POST(self):
        body, rid = json.loads(self.rfile.read(int(self.headers["Content-Length"]))), {"X-Request-Id": str(uuid.uuid4())}
        pre = {k: v for k, v in body.items() if k != "stream_options"}
        pre.update(max_tokens=1, stream=False, kv_transfer_params=dict(do_remote_decode=True, do_remote_prefill=False, remote_engine_id=None, remote_block_ids=None, remote_host=None, remote_port=None))
        try:
            kv = json.load(post_chat(dict(host="127.0.0.1", port=8100), pre, rid)).get("kv_transfer_params")
            relay(self, dict(host="127.0.0.1", port=8200), dict(body, kv_transfer_params=kv), rid)
        except Exception as e:
            d = json.dumps({"error": scrub(repr(e))}).encode(); self.send_response(502); self.send_header("Content-Length", str(len(d))); self.end_headers(); self.wfile.write(d)

ATTEMPTS = [("NixlConnector, default transport", {}, {}),
            ("NixlConnector, host staging buffer, UCX over tcp and shared memory", {"UCX_TLS": "sm,tcp,cuda_copy,self"}, {"kv_buffer_device": "cpu"})]
def try_disagg(limit_s=720):
    t_end = time.time() + limit_s
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nixl==1.5.2"], capture_output=True, text=True, timeout=400)
    if r.returncode: print("disaggregated serving did not run: nixl install failed:\n" + scrub(r.stderr[-1000:])); return
    for label, env, cfg in ATTEMPTS:
        if time.time() > t_end - 150: print("no time left for attempt:", label); break
        print(f"--- attempt: {label}")
        try:
            for n, gpu, port, role, side in (("pf", 0, 8100, "kv_producer", 5600), ("dc", 1, 8200, "kv_consumer", 5601)):
                kvc = json.dumps({"kv_connector": "NixlConnector", "kv_role": role, "kv_load_failure_policy": "fail", **cfg})
                launch(n, gpu, port, "--max-num-batched-tokens", "8192", "--kv-transfer-config", kvc, env={"VLLM_NIXL_SIDE_CHANNEL_PORT": str(side), **env})
            for n, port in (("pf", 8100), ("dc", 8200)): print(f"{n} ready after {wait_ready(n, port, 240):.0f} s")
            proxy = Srv(("127.0.0.1", 0), PD); threading.Thread(target=proxy.serve_forever, daemon=True).start()
            pp, ask = proxy.server_address[1], [{"role": "user", "content": "Name three colors."}]
            a, b = chat(pp, ask, 16, stream=False, ignore_eos=False), chat(8100, ask, 16, stream=False, ignore_eos=False)
            if a["outcome"] == "error": raise RuntimeError(f"request through the proxy failed: {a['err']}\n--- consumer log tail ---\n{log_tail('dc', 1500)}\n--- producer log tail ---\n{log_tail('pf', 1500)}")
            print(f"smoke test: proxy reply == producer's own reply: {a['text'] == b['text']}")
            interf("KV connector (NIXL)", pp, pp, 5); proxy.shutdown(); proxy.server_close()
            m = urllib.request.urlopen("http://127.0.0.1:8200/metrics", timeout=10).read().decode()
            print("consumer metrics that mention the transfer:", [l[:110] for l in m.splitlines() if not l.startswith("#") and re.search(r"nixl|kv_transfer|external", l)][:6])
            return
        except Exception as e: print("attempt failed with this error:\n" + scrub(str(e))[-2500:])
        finally: stop("pf", "dc")
    print("disaggregated serving did not run on this machine. The error above is the result.")

def part_d():
    stop("r1", "r2")
    launch("da", 0, 8101, "--max-num-batched-tokens", "8192"); launch("db", 1, 8102, "--max-num-batched-tokens", "8192")
    for n, p in (("da", 8101), ("db", 8102)): print(f"{n} ready after {wait_ready(n, p):.0f} s")
    chat(8101, [{"role": "user", "content": "hi"}], 8); chat(8102, [{"role": "user", "content": "hi"}], 8)
    interf("chats only (floor)", 8101, None, 1)
    interf("mixed, budget 8192", 8101, 8101, 2)
    interf("roles split, no KV transfer", 8101, 8102, 3)
    stop("da", "db"); launch("da", 0, 8101, "--max-num-batched-tokens", "512"); print(f"da (budget 512) ready after {wait_ready('da', 8101):.0f} s")
    chat(8101, [{"role": "user", "content": "hi"}], 8)
    interf("mixed, budget 512", 8101, 8101, 4); stop("da")
    if late(): print("budget used: the KV connector try is skipped")
    else: try_disagg(min(720, DEADLINE[0] - time.time()))
    df = pd.DataFrame(D_ROWS); show(df.round(3), "d_interference")
    f = df.iloc[0]
    for _, r in df.iloc[1:].iterrows(): print(f"{r.setup:30s} ITL p99 {r.itl_p99 / f.itl_p99:5.1f}x the floor, long-prompt TTFT p50 {r.long_ttft_p50:6.0f} ms")
    df.set_index("setup")[["itl_p50", "itl_p99", "long_ttft_p50"]].plot.bar(ylabel="ms", title="Chat inter-token latency and long-prompt TTFT"); plt.tight_layout(); plt.savefig(f"{OUT}/d_interference.png"); plt.close()
part("D prefill and decode interference", part_d, 1000)

## Part E: cold start (inference.1)

Question: how long does a new replica need before it answers, and what takes the time? Why the exam asks: this is the missed question about slow NIM pods. The best fix is to cache the model once on shared network storage that every new pod mounts (`cache.txt`). A readiness probe that says ready early does not help: traffic arrives at a pod that cannot answer.

Three starts of the same server on GPU 0:

1. **No local weights.** A new empty Hugging Face cache and a new vLLM cache. The server downloads the model and builds its compile cache.
2. **Cached weights, page cache dropped.** The weights are on disk. `posix_fadvise` asks the kernel to forget their cached pages, so the server reads them from disk. The table shows the read speed of the file before the start.
3. **Cached weights, warm page cache.** Right after state 2. The files are in RAM.

A client sends one request per second from the start of the process. The first answer sets the cold start time. The requests before it failed. They are the traffic that an early ready probe would lose.

The phases come from the vLLM log. Download, weights load and engine start each have a line. A line that this vLLM version does not print shows as `NaN`. The same cell also checks the KV cache memory per token against the log, for the term "context-length effects on GPU memory".

Shared network storage is not measured here. The notebook does the arithmetic from the measured times.

In [ ]:
# 10. Part E: cold start. Time to the first answer with no local weights, with cached weights and a cold page cache, and with a warm page cache.
E_COLD = {}      # seconds to the first answer, cold and cached: part F uses them as replica start times
SNAP = "hub/models--Qwen--Qwen2.5-1.5B-Instruct/snapshots/*/"
def snapshot_files(hf): return [os.path.realpath(p) for p in glob.glob(f"{hf}/{SNAP}*")]
def drop_page_cache(files):
    for f in files:
        fd = os.open(f, os.O_RDONLY); os.posix_fadvise(fd, 0, 0, os.POSIX_FADV_DONTNEED); os.close(fd)     # asks the kernel to forget the cached pages of this file
def read_gbps(path, nbytes=256 * 2 ** 20):
    """Read speed of the last 256 MB of a file. A cold page cache reads from disk. A warm one reads from RAM."""
    size = os.path.getsize(path); off, n, fd, t = max(0, size - nbytes), 0, os.open(path, os.O_RDONLY), time.perf_counter()
    while n < min(nbytes, size):
        b = os.pread(fd, 16 * 2 ** 20, off + n)
        if not b: break
        n += len(b)
    os.close(fd); return n / 2 ** 30 / (time.perf_counter() - t)

def cold_start(label, hf, vc, drop=False):
    files = snapshot_files(hf)
    if drop: drop_page_cache(files)
    big = max((f for f in files if f.endswith(".safetensors")), key=os.path.getsize, default=None)
    gbps = read_gbps(big) if big else float("nan")
    t0 = time.time(); launch("cold", 0, 8300, env={"HF_HOME": hf, "VLLM_CACHE_ROOT": vc})
    ready = first = None; fails = 0
    while first is None and time.time() - t0 < max(60, DEADLINE[0] - time.time()):
        if PROCS["cold"].poll() is not None: raise RuntimeError(f"server exited:\n{log_tail('cold')}")
        now = time.time() - t0
        if ready is None:
            try: urllib.request.urlopen("http://127.0.0.1:8300/health", timeout=2).close(); ready = now
            except Exception: pass
        if chat(8300, [{"role": "user", "content": "Say hi."}], 4, stream=False, ignore_eos=False)["outcome"] != "error": first = now
        else: fails += 1; time.sleep(1)       # a request sent while the pod cannot answer yet is lost
    log = open(f"{OUT}/cold.log").read(); stop("cold")
    return dict(state=label, read_gb_s=gbps, to_ready_s=ready, to_first_answer_s=first, lost_requests=fails, **parse_startup(log))

def part_e():
    global E_COLD
    stop("r1", "r2", "da", "db"); hf, vc = f"{OUT}/hf_cold", f"{OUT}/vllm_cache_cold"
    for d in (hf, vc): shutil.rmtree(d, ignore_errors=True)
    rows = []
    for label, drop in (("1 no local weights (download)", False), ("2 cached weights, page cache dropped", True), ("3 cached weights, warm page cache", False)):
        if late() and rows: print("budget used, stop before", label); break
        try: rows.append(cold_start(label, hf, vc, drop)); print(f"{label}: first answer after {rows[-1]['to_first_answer_s']} s")
        except Exception as e: print(f"state {label} failed:", scrub(e)[-1800:]); stop("cold"); break
    if not rows: return
    df = pd.DataFrame(rows).astype({"to_ready_s": float, "to_first_answer_s": float})
    df["other_s"] = df.to_first_answer_s - df[["download_s", "weights_load_s", "engine_init_s"]].fillna(0).sum(axis=1)
    show(df.drop(columns=["kv_tokens", "kv_gib"]).round(1), "e_cold_start")
    print("download_s, weights_load_s and engine_init_s come from the vLLM log. compile_s and graph_capture_s run inside engine_init_s. other_s is the rest (process start, imports, API server).")
    print(f"requests sent once per second while the server could not answer yet: {df.lost_requests.tolist()} were lost. A probe that says ready too early lets traffic in at that point. The first answer comes at the same time.")
    if len(df) > 1 and df.to_first_answer_s[:2].notna().all():
        E_COLD = dict(cold=df.to_first_answer_s[0], cached=df.to_first_answer_s[1])
        files, save = snapshot_files(hf), df.to_first_answer_s[0] - df.to_first_answer_s[1]; gib = sum(os.path.getsize(f) for f in files) / 2 ** 30
        print(f"\nWith the model cached before the pod starts, a pod answers {save:.0f} s sooner (state 1 minus state 2). Arithmetic, shared storage not measured here:")
        for n in (1, 4, 16): print(f"  {n:2d} pods from zero: {n * gib:5.1f} GiB downloaded without a shared cache, {gib:4.1f} GiB once with one. Each pod saves about {save:.0f} s.")
    # context length and GPU memory: bytes of KV cache per token, from the model config, against what the vLLM log says
    cfgs = glob.glob(f"{hf}/{SNAP}config.json")
    if cfgs and df.kv_gib.notna().any():
        c, r = json.load(open(cfgs[0])), df.dropna(subset=["kv_gib"]).iloc[0]
        bpt = 2 * c["num_hidden_layers"] * c["num_key_value_heads"] * (c.get("head_dim") or c["hidden_size"] // c["num_attention_heads"]) * 2
        print(f"\nKV cache per token = 2 x layers x kv heads x head dim x 2 bytes (fp16) = {bpt} bytes. vLLM log: {r.kv_tokens:,.0f} tokens in {r.kv_gib} GiB = {r.kv_gib * 2 ** 30 / r.kv_tokens:.0f} bytes per token.")
        print("formula and log agree within 6%:", abs(bpt * r.kv_tokens / 2 ** 30 - r.kv_gib) / r.kv_gib < 0.06)
        t = pd.DataFrame([dict(context_tokens=n, kv_gib_per_sequence=n * bpt / 2 ** 30, sequences_that_fit_fp16=r.kv_gib * 2 ** 30 // (n * bpt), sequences_if_fp32_arithmetic=r.kv_gib * 2 ** 30 // (n * bpt * 2)) for n in (1024, 4096, 32768)])
        show(t.round(3), "e_context_memory")
    else: print("the vLLM log has no KV cache lines (or no config.json): the context memory table is skipped")
part("E cold start", part_e, 540)

## Part F: queue, load shedding and autoscaling (inference.1)

Question: what does a traffic spike do to latency, and which fix works? Why the exam asks: request queuing, autoscaling, cold starts and scaling strategies decide whether users see errors or slow replies.

This is a simulation. The service times are the measured end-to-end times of part A (real-time, concurrency 8). Each replica has 8 slots. The replica start times are the measured times of part E. Load is 60% of the capacity of two replicas, then 180% for 100 s, then 60% again.

The simulator must be right before you use it. Three checks run first (cell 5): the mean wait of an M/M/1 queue, the mean wait of an M/M/8 queue (Erlang C formula) and the blocking probability of an M/M/1/K queue. Each compares the simulator with the formula.

Rules of the autoscaler: every 2 s, if more than 8 requests wait, start one more replica, up to 6. It pays for a replica from the moment it starts. A warm pool holds 2 loaded replicas that cost money all the time and attach in 2 s. The SLO is 3 times the median service time. A shed request counts as an SLO violation.

In [ ]:
# 11. Part F: a bounded queue, load shedding and autoscaling against a traffic spike. Service times are the measured ones from part A.
def part_f():
    svc_src = np.array(A_E2E) if len(A_E2E) > 20 else np.random.default_rng(0).lognormal(0.7, 0.4, 200)
    if len(A_E2E) <= 20: print("part A gave no service times: using a lognormal stand-in with a mean of about 2.2 s")
    cold, cached = E_COLD.get("cold", 180.0), E_COLD.get("cached", 60.0)
    if not E_COLD: print("part E gave no start times: using 180 s (download) and 60 s (cached) as stated assumptions")
    slots, n0, rng = 8, 2, np.random.default_rng(7)
    cap = n0 * slots / svc_src.mean()                                  # requests per second that 2 replicas can finish
    arr = np.sort(np.concatenate([rng.uniform(s, e, rng.poisson(cap * f * (e - s))) for s, e, f in ((0, 100, 0.6), (100, 200, 1.8), (200, 400, 0.6))]))
    svc, slo = rng.choice(svc_src, len(arr)), 3 * float(np.median(svc_src))
    print(f"{len(arr)} requests in 400 s. Load: 60% of capacity ({cap:.1f} req/s), then 180% for 100 s, then 60%. SLO: latency under {slo:.1f} s (3 x the median service time). Replica start: cold {cold:.0f} s, cached {cached:.0f} s.")
    sc = lambda d, pool=0: dict(tick=2, thresh=slots, delay=d, warm=2, max=6, pool=pool)
    cases = [("fixed 2 replicas, unbounded queue", {}), ("fixed 2, queue limit 32 (shed)", dict(qmax=32)), (f"autoscale, cold start {cold:.0f} s", dict(scale=sc(cold))),
             (f"autoscale, cached weights {cached:.0f} s", dict(scale=sc(cached))), ("autoscale, warm pool of 2 (2 s)", dict(scale=sc(cached, 2))),
             ("autoscale, warm pool of 2, queue limit 32", dict(scale=sc(cached, 2), qmax=32))]
    rows, lats = [], {}
    for name, kw in cases:
        s = simulate(arr, svc, slots, n0, **kw); done = ~s["shed"]; lat = s["lat"][done]; lats[name] = s["lat"]
        bad = int(s["shed"].sum() + (lat > slo).sum())
        rows.append(dict(setup=name, shed_pct=100 * s["shed"].mean(), p50_s=pct(lat, 50), p95_s=pct(lat, 95), p99_s=pct(lat, 99), slo_violations=rate(bad, len(arr)), peak_queue=s["peak"], replica_seconds=s["replica_s"]))
    df = pd.DataFrame(rows); show(df.round(1), "f_scaling")
    best = df.loc[df.slo_violations.str.extract(r"= ([\d.]+)%")[0].astype(float).idxmin()]
    print(f"fewest SLO violations: {best.setup} ({best.slo_violations}) at {best.replica_seconds:.0f} replica-seconds. The first row costs {df.replica_seconds[0]:.0f}.")
    for name in list(lats)[:1] + list(lats)[2:5]:
        b = np.arange(0, 400, 10); m = [np.nanmean(lats[name][(arr >= x) & (arr < x + 10)]) for x in b]; plt.plot(b, m, label=name)
    plt.axvspan(100, 200, alpha=0.1); plt.axhline(slo, ls=":", c="k"); plt.xlabel("arrival time s"); plt.ylabel("mean latency s"); plt.legend(fontsize=7); plt.tight_layout(); plt.savefig(f"{OUT}/f_scaling.png"); plt.close()
part("F queue and scaling", part_f, 90, always=True)

## Part G: Triton dynamic batcher against sequence batcher (inference.1)

Question: a stateful anomaly model gets windows from each machine. Every window of one machine must reach the same model instance. Which batcher? Why the exam asks: this is the missed question about Triton (`batcher.txt`).

This is a plain Python simulation of two model instances. The model keeps a running score for each machine inside the instance. The truth is the score from a single instance that sees the windows in order. A **wrong-state answer** is an output that differs from that truth. A stateless model is a control: its output only depends on the window.

- **Dynamic batcher**: any free instance takes the next windows from one shared queue, up to a batch of 8. All rows of a batch read the state from before the batch.
- **Sequence batcher**: the first window of a sequence picks an instance. The batcher sends every later window with the same correlation ID to that instance. A batch holds at most one window per sequence. The slot frees at the end of the sequence.

40 machines, 25 windows each, 1,000 windows, three seeds.

In [ ]:
# 12. Part G: Triton dynamic batcher against sequence batcher, two stateful model instances. 40 machines, 25 windows each = 1,000 windows.
def part_g():
    rows = []
    for name, b, st in (("dynamic batcher, stateful model", "dynamic", True), ("sequence batcher, stateful model", "sequence", True), ("dynamic batcher, stateless model", "dynamic", False)):
        for seed in (0, 1, 2):
            r = triton_sim(b, st, seed=seed); lo, hi = wilson(r["wrong"], r["windows"])
            rows.append(dict(setup=name, seed=seed, wrong_per_1000=1000 * r["wrong"] / r["windows"], ci95=f"[{1000 * lo:.0f}, {1000 * hi:.0f}]", mean_batch=r["batch"], steps=r["steps"]))
    df = pd.DataFrame(rows); show(df.round(1), "g_triton")
    m = df.groupby("setup", sort=False)[["wrong_per_1000", "mean_batch", "steps"]].mean()
    print(m.round(1).to_string())
    d, s = m.loc["dynamic batcher, stateful model"], m.loc["sequence batcher, stateful model"]
    print(f"stateful model: the dynamic batcher gave {d.wrong_per_1000:.0f} wrong answers per 1,000 windows, the sequence batcher {s.wrong_per_1000:.0f}. Mean batch size {d.mean_batch:.1f} against {s.mean_batch:.1f}.")
part("G Triton batchers", part_g, 30, always=True)

In [ ]:
# 13. Stop all servers, print GPU memory, and the status of every part.
stop(*PROCS)
print("GPU memory in use (MiB):", gpu_mib())
st = pd.DataFrame(STATUS); st.to_csv(f"{OUT}/status.csv", index=False); print(st.to_string(index=False))
import textwrap
print(f"total {(time.time() - T0) / 60:.1f} min. Saved in {OUT}:\n" + textwrap.fill(", ".join(sorted(f for f in os.listdir(OUT) if f.endswith((".csv", ".png", ".json")))), 150))

## What to look for

- All times are in ms unless the column name ends in `_s`.
- **Part A.** Compare tokens/s of the batch with real-time at the same 64 sequences. The gap is the cost of the HTTP path and of one request per call. Compare TTFT of the stream with the end-to-end time of the full reply. Check the three client numbers: a client bottleneck would make every mode look slower than the server really is. `short` and `error` counts must be 0. A non-zero count is an outcome of its own.
- **Part B.** Read `ttft_turn2plus_p50` and `cache_hit_rate` together. The first turn of a document cannot hit the cache for its own text, so turns 2 to 4 show the policy. Check `share_on_r1`: a policy that sticks to one replica can unbalance the load. The paired interval tells you if the TTFT gap is larger than the noise. The router records only what it sent, not what the engine evicted. The Wilson interval on the hit rate counts each token as one trial, so it is too narrow. The paired TTFT interval is the honest one. Limits: 2 repeats, one model, a router that does not see engine events.
- **Part C.** `failed` has a Wilson interval. `retries` counts extra tries. Retry alone saves requests but pays the failed try each time. The health check removes that cost after about one poll. In-flight requests that were already streaming cannot be retried. This run uses non-streaming requests.
- **Part D.** Read `itl_p99` and `stalls_over_100ms` against the floor. A smaller token budget trades chat smoothness for a slower long prefill. The role split removes the interference with no cache transfer. The KV connector row exists only if the connector ran. If not, the printed error is the result. The proxy hop is in the connector row only.
- **Part E.** State 1 against state 2 shows what the download costs. State 2 against state 3 shows what the disk read costs. `lost_requests` is the traffic that an early ready probe would lose. A `NaN` in a phase column means this vLLM version prints no line for it. The context table is arithmetic from the model config. The `fp32` column is not measured.
- **Part F.** Compare the cold-start row with the cached and warm-pool rows. A replica that arrives after the spike does not help. A queue limit trades slow replies for rejected requests. The warm pool pays replica-seconds for fewer violations. The result depends on the stated rules: load levels, 8 slots, SLO, tick, and the start times.
- **Part G.** The sequence batcher keeps the batch size above 1, so it still batches. The stateless control shows why the dynamic batcher is right for a model without state.
- Text lines that start with "highest", "lowest", "fewest" or "the client is" come from the numbers of that run.

## Exam rules this proves

1. **inference.4**: Batch work asks for throughput and has no per-request wait. Real-time work asks for a low end-to-end time. Streaming work asks for a low time to the first token. Take that time from the first chunk that holds text. (Part A)
2. **inference.1**: The engine runs the model on a GPU. Dynamo runs the fleet of engines. It adds disaggregated serving, KV cache-aware routing, KV cache offload across memory tiers and SLA-based autoscaling. It does not replace vLLM, SGLang or TensorRT-LLM. This is the missed Dynamo question. (Part B, `introduction.txt`)
3. **inference.1**: KV-aware routing sends a prompt to the replica that most likely holds its prefix. Prove it with cached prompt tokens and with TTFT on the later turns of a chat. (Part B)
4. **inference.1**: A gateway hides the provider behind one call. Retry works only before a byte reaches the client. A health check stops traffic to a dead provider. A hosted model is the last provider in the list. (Part C)
5. **inference.3**: Prefill and decode compete on one GPU, and a long prompt stalls the chats that decode. Smaller chunks, separate GPUs or disaggregated serving with a KV connector limit the stall. The vLLM documentation says that disaggregated prefill does not improve throughput (page "Disaggregated Prefilling", read 2026-11-25). (Part D)
6. **inference.1**: A cold start is download, weights load and engine setup. Cache the model once on shared network storage that every new pod mounts. A probe that says ready early only sends traffic to a pod that cannot answer. This is the missed NIM question. (Part E, `cache.txt`)
7. **inference.1**: A bounded queue turns overload into fast rejections. Autoscaling helps only if the new replica is ready before the spike ends. A cached model or a warm pool shortens that wait. (Part F)
8. **inference.1**: A stateful model needs the sequence batcher. It keeps every request with the same correlation ID on one instance and still batches. The dynamic batcher is for stateless models. This is the missed Triton question. (Part G, `batcher.txt`)
9. **inference.3**: KV cache per token is 2 x layers x kv heads x head dim x bytes per value. Context length times the number of sequences sets the KV memory. (Part E, `mastering-llm-techniques-inference-optimization.txt`)

### The exam terms of inference.1, inference.3 and inference.4

| Term | One plain line | Where shown |
|---|---|---|
| Inference-serving infrastructure | Everything between the user and the GPU: gateway, servers, queue, scaler. | Whole notebook |
| Model servers | Processes that load a model and answer requests: here two vLLM replicas, and the simulated Triton instances of part G. | A, B |
| Gateways and routing | One endpoint that picks a server for each request: rr, lor, prefix. | B |
| Provider abstraction | One client call over local replicas and a hosted model. | C |
| NVIDIA Dynamo | The layer above the engines: disaggregated serving, KV-aware routing, KV offload, SLA-based autoscaling. | B (routing), D (disaggregation), F (scaling idea) |
| Autoscaling | The replica count follows the load. | F |
| Request queuing | A request waits for a free slot. A bounded queue sheds extra load. | F |
| Cold starts | Time from process start to first answer. | E |
| Scaling strategies | Fixed, autoscale with cold start, with cached weights, with a warm pool, with shedding. | F |
| Optimization techniques | The group of the terms below. | B, D, E |
| KV caching | Keep keys and values of earlier tokens. The formula and the vLLM log agree on the bytes per token. | E |
| Prefix caching | Reuse the KV cache of a shared prompt start. Cached tokens come from `/metrics`. | B |
| Disaggregated serving | Prefill and decode on different GPUs with a KV connector. A bounded try. | D |
| Context-length effects on GPU memory | KV bytes per sequence grow with the context. The table shows 1k, 4k and 32k tokens. | E |
| Precision tradeoffs | These runs use fp16 because the T4 has no bf16. The context table shows what fp32 would cost (arithmetic). | E |
| Quantization | Not run here. See `vllm-t4-7b-tensor-parallel-vs-awq.ipynb` in this repository. | not run here |
| Speculative decoding | Not run here. See `vllm-t4-inference-optimizations.ipynb`. | not run here |
| Tensor parallelism | Not run here. See `vllm-t4-7b-tensor-parallel-vs-awq.ipynb`. | not run here |
| Serving best practices | Pick the mode by the number that matters: tokens/s, end-to-end time or TTFT. | A |
| Batch | All requests at once, throughput only. | A |
| Real-time | One full reply per request. | A |
| Streaming services | Tokens as they come. TTFT from the first chunk with text. | A |
| Triton dynamic batcher and sequence batcher | Stateless against stateful models. | G |